## 1. Load Transition Matrix

In this section, I load the transition matrix generated from SQL Server.

The transition matrix contains:

- current_aid
- current_type
- next_aid
- next_type
- transition_count

This matrix summarizes user navigation behavior and will be used to build recommendation dictionaries for clicks, carts, and orders.

In [ ]:
import pandas as pd

transition = pd.read_csv("C:/Users/minhn/Downloads/project2_otto-recommender-system/transition_summary.csv")

In [26]:
print(transition.shape)
transition.columns

(26119830, 5)


Index(['current_aid', 'current_type', 'next_aid', 'next_type',
       'transition_count'],
      dtype='object')

In [28]:
transition.head()

,current_aid,current_type,next_aid,next_type,transition_count
0,724121,clicks,1672025,clicks,10
1,1576965,clicks,1816561,clicks,1
2,507182,clicks,260456,clicks,1
3,353294,clicks,353294,clicks,3
4,77614,clicks,38877,clicks,2


In [17]:
transition["current_aid"] = transition["current_aid"].astype("int32")
transition["next_aid"] = transition["next_aid"].astype("int32")
transition["transition_count"] = transition["transition_count"].astype("int32")

transition["current_type"] = transition["current_type"].astype("category")
transition["next_type"] = transition["next_type"].astype("category")

In [24]:
transition.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26119830 entries, 0 to 26119829
Data columns (total 5 columns):
 #   Column            Dtype   
---  ------            -----   
 0   current_aid       int32   
 1   current_type      category
 2   next_aid          int32   
 3   next_type         category
 4   transition_count  int32   
dtypes: category(2), int32(3)
memory usage: 348.7 MB


## 2. Create Action-Specific Transition Tables

The OTTO competition evaluates three action types separately:

- Clicks
- Carts
- Orders

Therefore, I build three independent recommendation dictionaries using:

- next_type = clicks
- next_type = carts
- next_type = orders

In [33]:
click_df = transition[
    transition["next_type"] == "clicks"
].copy()

cart_df = transition[
    transition["next_type"] == "carts"
].copy()

order_df = transition[
    transition["next_type"] == "orders"
].copy()

print(click_df.shape)
print(cart_df.shape)
print(order_df.shape)

(24295495, 5)
(1089822, 5)
(734513, 5)


## 3. Build Recommendation Dictionaries

For each product, I keep the Top 20 most frequent next products.

The transition count is used as a proxy for recommendation strength.

In [64]:
def build_recommendation_dict(df):

    recommendation_dict = {}

    df = df.sort_values(
        [
            "current_aid",
            "current_type",
            "transition_count"
        ],
        ascending=[True, True, False]
    )

    grouped = df.groupby(
        ["current_aid", "current_type"]
    )

    for key, group in grouped:

        recommendation_dict[key] = list(
            zip(
                group["next_aid"].head(20),
                group["transition_count"].head(20)
            )
        )

    return recommendation_dict

### Build 3 dictionaries

In [66]:
click_dict = build_recommendation_dict(click_df)

cart_dict = build_recommendation_dict(cart_df)

order_dict = build_recommendation_dict(order_df)

C:\Users\minhn\AppData\Local\Temp\ipykernel_9944\1030161307.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby(


### Check example

In [72]:
list(click_dict.items())[:3]

[((0, 'clicks'),
  [(937063, 1),
   (994505, 1),
   (99474, 1),
   (1306175, 1),
   (475558, 1),
   (817833, 1),
   (1735605, 1),
   (643097, 1),
   (676735, 1),
   (218900, 1),
   (1012453, 1),
   (320565, 1),
   (13759, 1),
   (1627540, 1)]),
 ((1, 'clicks'),
  [(161916, 1),
   (166683, 1),
   (504137, 1),
   (1835190, 1),
   (606800, 1),
   (1771555, 1)]),
 ((2, 'clicks'), [(504438, 1), (827690, 1), (1716719, 1)])]

## 4. Load Test Sessions

The OTTO test set contains truncated sessions.

Task is to predict future clicks, carts, and orders after the truncation point.

In [54]:
import json
import pandas as pd

sessions = []

with open("C:/Users/minhn/Downloads/project2_otto-recommender-system/dataset/test.jsonl", "r") as f:

    for line in f:

        row = json.loads(line)

        session_id = row["session"]

        events = [
            (
                e["aid"],
                e["type"]
            )
            for e in row["events"]
        ]

        sessions.append({
            "session": session_id,
            "events": events
        })

test_df = pd.DataFrame(sessions)

In [56]:
test_df.head()

,session,events
0,12899779,"[(59625, clicks)]"
1,12899780,"[(1142000, clicks), (582732, clicks), (973453,..."
2,12899781,"[(141736, clicks), (199008, clicks), (57315, c..."
3,12899782,"[(1669402, clicks), (1494780, clicks), (149478..."
4,12899783,"[(255297, clicks), (1114789, clicks), (255297,..."


In [60]:
test_df.iloc[1]["events"]

[(1142000, 'clicks'),
 (582732, 'clicks'),
 (973453, 'clicks'),
 (736515, 'clicks'),
 (1142000, 'clicks')]

## 5. Session-Based Recommendation

Instead of using only the last interacted product, I use all products within the session.

Each product contributes recommendation candidates through the transition matrix.

Candidates are ranked by accumulated transition counts.

In [82]:
from collections import defaultdict

def recommend_session(
    events,
    recommendation_dict,
    top_k=20
):

    scores = defaultdict(float)

    unique_events = list(
        reversed(
            list(dict.fromkeys(events))
        )
    )

    for event in unique_events:

        neighbors = recommendation_dict.get(
            event,
            []
        )

        for next_aid, transition_count in neighbors:

            scores[next_aid] += transition_count

    recommendations = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        aid
        for aid, score
        in recommendations[:top_k]
    ]

In [88]:
sample_events = test_df.iloc[0]["events"]

print(sample_events[:5])

print(
    recommend_session(
        sample_events,
        click_dict
    )
)

[(59625, 'clicks')]
[737445, 499621, 1804863]


## 6. Generate Predictions

For each session:

- Generate click recommendations
- Generate cart recommendations
- Generate order recommendations

Each session contributes three rows to the final submission file.

In [93]:
print(test_df.shape)

(1671803, 2)


In [95]:
sample_test = test_df.head(1000)

In [97]:
import time

start = time.time()

predictions = []

for row in sample_test.itertuples(index=False):

    session_id = row.session
    events = row.events

    click_preds = recommend_session(
        events,
        click_dict
    )

    cart_preds = recommend_session(
        events,
        cart_dict
    )

    order_preds = recommend_session(
        events,
        order_dict
    )

    predictions.append({
        "session_type":
            f"{session_id}_clicks",
        "labels":
            " ".join(map(str, click_preds))
    })

    predictions.append({
        "session_type":
            f"{session_id}_carts",
        "labels":
            " ".join(map(str, cart_preds))
    })

    predictions.append({
        "session_type":
            f"{session_id}_orders",
        "labels":
            " ".join(map(str, order_preds))
    })

print(
    "Seconds:",
    round(time.time() - start, 2)
)

Seconds: 0.28


In [99]:
pip install tqdm

In [101]:
from tqdm import tqdm

predictions = []

for row in tqdm(
    test_df.itertuples(index=False),
    total=len(test_df)
):

    session_id = row.session
    events = row.events

    click_preds = recommend_session(
        events,
        click_dict
    )

    cart_preds = recommend_session(
        events,
        cart_dict
    )

    order_preds = recommend_session(
        events,
        order_dict
    )

    predictions.append({
        "session_type":
            f"{session_id}_clicks",
        "labels":
            " ".join(map(str, click_preds))
    })

    predictions.append({
        "session_type":
            f"{session_id}_carts",
        "labels":
            " ".join(map(str, cart_preds))
    })

    predictions.append({
        "session_type":
            f"{session_id}_orders",
        "labels":
            " ".join(map(str, order_preds))
    })

100%|██████████| 1671803/1671803 [03:04<00:00, 9082.63it/s] 


In [104]:
submission = pd.DataFrame(
    predictions
)

submission.head()

,session_type,labels
0,12899779_clicks,737445 499621 1804863
1,12899779_carts,
2,12899779_orders,
3,12899780_clicks,736515 1142000 582732 1712906 973453 1502122 4...
4,12899780_carts,736515 582732 1142000 973453 1303148 595830 16...


In [106]:
submission.shape

(5015409, 2)

In [108]:
sample_events = test_df.iloc[0]["events"]

print(
    recommend_session(
        sample_events,
        click_dict
    )
)

print(
    recommend_session(
        sample_events,
        cart_dict
    )
)

print(
    recommend_session(
        sample_events,
        order_dict
    )
)

[737445, 499621, 1804863]
[]
[]


## Some problems on the dataset

### Handling Sparse Transition Data

The transition matrix is highly sparse.

Analysis shows that more than 75% of transitions occur only once in the training data.

As a result, many session-product combinations do not have corresponding cart or order transitions.

For example:

Current State:
(59625, clicks)

Click Recommendations:
[737445, 499621, 1804863]

Cart Recommendations:
[]

Order Recommendations:
[]

This creates empty predictions for cart and order tasks, which negatively impacts Recall@20.

To address this issue, a popularity-based fallback strategy is introduced.

Whenever no transition-based recommendation is available, the model supplements predictions using globally popular products ranked by total transition frequency.

Benefits:

- Prevents empty prediction lists
- Improves coverage across all sessions
- Provides a strong baseline for sparse products
- Typically improves Recall@20 for carts and orders

This hybrid approach combines:

1. Transition-based recommendations (personalized)
2. Popularity-based recommendations (fallback)

to ensure every session receives up to 20 candidate products.

## 7.1 Popular Product Fallback

Many products do not have sufficient cart or order transitions.

To avoid empty predictions, a global popularity fallback is introduced.

Popular products are ranked by total transition frequency and used whenever no recommendations are available.

In [113]:
popular_products = (
    transition
    .groupby("next_aid")["transition_count"]
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

print(len(popular_products))

1561666


In [115]:
popular_products[:20]

[29735,
 1733943,
 108125,
 1460571,
 832192,
 1603001,
 184976,
 1083665,
 166037,
 322370,
 554660,
 959208,
 247240,
 332654,
 1502122,
 231487,
 673407,
 1196256,
 986164,
 508883]

## 7.2 Enhanced Recommendation Function

If no transition candidates are found, the model falls back to globally popular products.

This guarantees that every session receives up to 20 recommendations for clicks, carts and orders.

In [120]:
from collections import defaultdict

In [122]:
def recommend_session(
    events,
    recommendation_dict,
    popular_products,
    top_k=20
):

    scores = defaultdict(float)

    unique_events = list(
        reversed(
            list(dict.fromkeys(events))
        )
    )

    for event in unique_events:

        neighbors = recommendation_dict.get(
            event,
            []
        )

        for next_aid, transition_count in neighbors:

            scores[next_aid] += transition_count

    recommendations = [
        aid
        for aid, score
        in sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
    ]

    if len(recommendations) < top_k:

        for aid in popular_products:

            if aid not in recommendations:

                recommendations.append(aid)

            if len(recommendations) >= top_k:

                break

    return recommendations[:top_k]

In [124]:
sample_events = test_df.iloc[0]["events"]

print(
    recommend_session(
        sample_events,
        click_dict,
        popular_products
    )
)

print(
    recommend_session(
        sample_events,
        cart_dict,
        popular_products
    )
)

print(
    recommend_session(
        sample_events,
        order_dict,
        popular_products
    )
)

[737445, 499621, 1804863, 29735, 1733943, 108125, 1460571, 832192, 1603001, 184976, 1083665, 166037, 322370, 554660, 959208, 247240, 332654, 1502122, 231487, 673407]
[29735, 1733943, 108125, 1460571, 832192, 1603001, 184976, 1083665, 166037, 322370, 554660, 959208, 247240, 332654, 1502122, 231487, 673407, 1196256, 986164, 508883]
[29735, 1733943, 108125, 1460571, 832192, 1603001, 184976, 1083665, 166037, 322370, 554660, 959208, 247240, 332654, 1502122, 231487, 673407, 1196256, 986164, 508883]


In [128]:
click_preds = recommend_session(
    events,
    click_dict,
    popular_products
)

In [130]:
cart_preds = recommend_session(
    events,
    cart_dict,
    popular_products
)

In [132]:
order_preds = recommend_session(
    events,
    order_dict,
    popular_products
)

In [136]:
predictions = []

for row in tqdm(
    test_df.itertuples(index=False),
    total=len(test_df)
):

    session_id = row.session
    events = row.events

    click_preds = recommend_session(
        events,
        click_dict,
        popular_products
    )

    cart_preds = recommend_session(
        events,
        cart_dict,
        popular_products
    )

    order_preds = recommend_session(
        events,
        order_dict,
        popular_products
    )

    predictions.append({
        "session_type": f"{session_id}_clicks",
        "labels": " ".join(map(str, click_preds))
    })

    predictions.append({
        "session_type": f"{session_id}_carts",
        "labels": " ".join(map(str, cart_preds))
    })

    predictions.append({
        "session_type": f"{session_id}_orders",
        "labels": " ".join(map(str, order_preds))
    })

100%|██████████| 1671803/1671803 [03:45<00:00, 7415.31it/s]


In [139]:
submission = pd.DataFrame(predictions)

In [141]:
submission.head(6)

,session_type,labels
0,12899779_clicks,737445 499621 1804863 29735 1733943 108125 146...
1,12899779_carts,29735 1733943 108125 1460571 832192 1603001 18...
2,12899779_orders,29735 1733943 108125 1460571 832192 1603001 18...
3,12899780_clicks,736515 1142000 582732 1712906 973453 1502122 4...
4,12899780_carts,736515 582732 1142000 973453 1303148 595830 16...
5,12899780_orders,736515 582732 1142000 973453 1758603 717070 16...


## Export

In [143]:
submission.to_csv(
    "submission.csv",
    index=False
)